## Differential Evolution


In this session, we will discover how Differential Evolution (DE) works. We will explore the underlying principles, implement core components from scratch, apply DE to classic benchmark functions, visualize its dynamics through a generated GIF, and tackle constraints.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from typing import Callable
from dataclasses import dataclass

### Optimization Problems

This cell defines three common benchmark functions, Sphere, Rosenbrock, and Rastrigin, used to test optimization algorithms. We also used these functions earlier to evaluate Adam, Momentum, and CMA-ES.


In [3]:
def sphere(x: np.ndarray) -> float:
    return float(np.sum(x**2))


def rosenbrock(x: np.ndarray) -> float:
    return float(np.sum(100.0 * (x[1:] - x[:-1] ** 2.0) ** 2.0 + (1.0 - x[:-1]) ** 2.0))


def rastrigin(x: np.ndarray) -> float:
    A: float = 10.0
    return float(A * len(x) + np.sum(x**2 - A * np.cos(2 * np.pi * x)))


BOUNDS = [(-5, 5), (-5, 5)]

### Differential Evolution

Differential Evolution is a simple yet powerful **population-based search algorithm** for continuous optimization. At every iteration it maintains a population of $pop\_size$ candidate solutions, each an $n$-dimensional real-valued vector. Three straightforward operators: mutation, crossover, and selection, are applied to push the population toward regions of lower objective value.

#### 1. Initialisation

Randomly sample $pop\_size$ vectors inside the problem's bounds.

#### 2. Mutation

For each target vector $x_i$:

1. Select three distinct solutions $a$, $b$, $c$ (none equal to $x_i$) from current population.
2. Create the mutant  
   $$v = a + F\,(b - c)$$
   where $F \in (0.4,1^+)$ is the scaling factor. The difference $(b-c)$ supplies direction and scale; $F$ stretches or shrinks the step.

#### 3. Crossover

Blend the mutant $v$ with its target $x_i$ to form the trial vector \(u\):

- For each coordinate $j$, copy $v_j$ into $u_j$ with probability $CR$; otherwise copy $x_{i,j}$.
- Force at least one coordinate to come from $v$ so that $u \neq x_i$.

Here $CR \in [0,1]$ is the _crossover rate_: higher $CR$ means aggressive mixing; lower $CR$ leaves the mutant mostly intact.

#### 4. Selection

Evaluate the objective $f(\cdot)$: $u \text{ replaces } x_i \quad \text{if} \quad f(u) < f(x_i)$

Greedy replacement ensures the population not deteriorates.

#### 5. Iteration & Termination

After every target has produced a trial and selection is done, the new population is complete. Repeat until a generation limit, a fitness threshold, or a stall criterion is reached.

#### 6. Parameter intuition

| Parameter                 | Typical range |
| ------------------------- | ------------- |
| Differential weight \(F\) | 0.4 – 1.0     |
| Crossover rate \(CR\)     | 0.1 – 0.9     |
| Population size \(P\)     | $5n$ – $10n$  |


In [4]:
@dataclass
class DEResult:
    best_vector: np.ndarray
    best_value: float
    history: list[np.ndarray]  # History of populations for animation


def differential_evolution(
    func: Callable[[np.ndarray], float],
    bounds: list[tuple[float, float]] = BOUNDS,
    pop_size: int = 50,
    F: float = 0.8,
    CR: float = 0.9,
    max_gen: int = 100,
) -> DEResult:
    """
    Implements the Differential Evolution algorithm for global optimization.

    Parameters:
        func: Objective function to minimize. Takes a numpy array and returns a float.
        bounds: list of (min, max) pairs for each dimension.
        pop_size: Number of individuals in the population.
        F: Mutation factor (typically in [0.4, 1.0]).
        CR: Crossover rate (typically in [0, 1]).
        max_gen: Maximum number of generations to evolve.
    """
    result = DEResult(best_vector=None, best_value=float("inf"), history=[])
    dimensions = len(bounds)
    lower_bounds = np.array([b[0] for b in bounds])
    upper_bounds = np.array([b[1] for b in bounds])

    # --- TODO: Exercise 1 - Initialization ---
    # Initialize the population: Create pop_size individuals.
    # Each individual is a vector of 'dimensions' length.
    # Use uniform distribution to sample the initial population.
    # --- End Exercise 1 ---
    population = np.random.uniform(lower_bounds, upper_bounds, (pop_size, dimensions))
    fitness = np.asarray([func(ind) for ind in population])
    result.history.append(population.copy())

    for generation in range(max_gen):
        ...

        for i in range(pop_size):
            ...

            # --- TODO: Exercise 2 - Mutation ---
            # Select three distinct random individuals (a, b, c) from the population,
            # different from the current target_vector at index i.
            # Ensure indices are unique and not equal to i.
            # Create the mutant vector: v = a + F * (b - c)
            # Handle boundary constraints for the mutant vector.
            # --- End Exercise 2 ---
            idxs = [idx for idx in range(pop_size) if idx != i]
            a, b, c = population[np.random.choice(idxs, 3, replace=False)]
            mutant = np.clip(a + F * (b - c), min=lower_bounds, max=upper_bounds)

            # --- TODO: Exercise 3 - Crossover ---
            # Create the trial vector by performing crossover between the target_vector and the mutant_vector.
            # For each dimension j:
            #   Pick a random number r from [0,1).
            #   If r < CR or j == j_rand (where j_rand is a randomly chosen dimension index),
            #     trial_vector[j] = mutant_vector[j]
            #   Else:
            #     trial_vector[j] = target_vector[j]
            # Ensure at least one dimension comes from the mutant vector (j_rand ensures this).
            # --- End Exercise 3 ---
            cross_points = np.random.rand(dimensions) < CR
            if not np.any(cross_points):
                j_rand = np.random.randint(dimensions)
                cross_points[j_rand] = True
            trial_vector = np.where(cross_points, mutant, population[i])

            # --- TODO: Exercise 4 - Selection ---
            # Evaluate the trial vector.
            # If the trial vector is better than or equal to the target vector,
            #   replace the target vector with the trial vector in the new_population and update its fitness.
            # Else, keep the target vector.
            # --- End Exercise 4 ---
            f_trial = func(trial_vector)
            f_target = fitness[i]
            if f_trial < f_target:
                population[i] = trial_vector
                fitness[i] = f_trial

        result.history.append(population.copy())

    result.best_vector = population[np.argmin(fitness)]
    result.best_value = func(result.best_vector)

    return result

### Test implemented DE


In [5]:
result = differential_evolution(sphere, bounds=BOUNDS, pop_size=50)

### Visualizing Search Dynamics


In [ ]:
def animate_de(
    func: Callable[[np.ndarray], float],
    history: list[np.ndarray],
    bounds: list[tuple[float, float]] = BOUNDS,
    frames: int | None = None,
    filename: str = "de_animation.gif",
) -> None:
    """
    Creates and saves a GIF showing how the DE population moves over generations.
    Only shows current generation points.
    """
    if frames is None:
        frames = len(history)

    assert len(bounds) == 2, (
        "This function only supports 2D visualization (expected 2 bounds)."
    )
    x_bounds = (bounds[0][0], bounds[0][1])
    y_bounds = (bounds[1][0], bounds[1][1])

    x = np.linspace(x_bounds[0], x_bounds[1], 200)
    y = np.linspace(y_bounds[0], y_bounds[1], 200)
    X, Y = np.meshgrid(x, y)
    coords = np.vstack([X.ravel(), Y.ravel()]).T
    Z = np.array([func(pt) for pt in coords]).reshape(X.shape)

    fig, ax = plt.subplots(figsize=(8, 6))
    contour = ax.contourf(X, Y, Z, levels=20, cmap="viridis")
    fig.colorbar(contour, ax=ax)

    scatter = ax.scatter([], [], s=20, color="red")  # <-- Keep reference

    def init():
        scatter.set_offsets(np.empty((0, 2)))
        return (scatter,)

    def update(i: int):
        ax.set_title(f"Generation {i}")
        pop = history[i]
        scatter.set_offsets(pop[:, :2])  # Update scatter points
        return (scatter,)

    ax.set_xlim(x_bounds[0], x_bounds[1])
    ax.set_ylim(y_bounds[0], y_bounds[1])

    anim = animation.FuncAnimation(
        fig, update, init_func=init, frames=frames, interval=200, blit=True
    )

    writer = animation.PillowWriter(fps=5)
    anim.save("plots/" + filename, writer=writer)
    plt.close(fig)
    print(f"Animation saved to {filename}")


In [9]:
animate_de(func=sphere, history=result.history)

Animation saved to de_animation.gif


### Experiments

Run DE on all three problems: Sphere, Rosenbrock and Rastrigin. For each problem:

- Generate and analyze convergence plots showing the progression of the best fitness value over iterations.
- Visualize the population dynamics over time to illustrate how the search space is explored and exploited.


In [24]:
for problem in [sphere, rosenbrock, rastrigin]:
    result = differential_evolution(problem, bounds=BOUNDS, pop_size=50)
    animate_de(func=problem, history=result.history, filename=f"{problem.__name__}.gif")

Animation saved to sphere.gif
Animation saved to rosenbrock.gif
Animation saved to rastrigin.gif


### Hyperparameters in Differential Evolution

Analyze the role and impact of the two key hyperparameters in Differential Evolution: the scaling factor ($F$) and the crossover rate ($CR$).

- What is the primary purpose of each hyperparameter in the context of the DE algorithm?
- How do variations in their values influence the algorithm’s exploration and exploitation behavior?
- Provide examples or illustrations, if possible, to support your analysis.


##### Scale Factor ($F$)

The scale factor (F) is a crucial control parameter that determines the step size of the search during optimization. <br>
It influences the population diversity and convergence speed by controlling the perturbation of the difference vector during the mutation process.<br>
A larger F value generally leads to wider exploration of the search space, while a smaller value favors more focused exploitation


##### Crossover Rate ($CR$)

The crossover rate (Cr) is a parameter that determines how many components of a trial vector are inherited from a mutant vector,<br>
and how many are inherited from the current vector.


In [25]:
parameters = {
    "F": [0.2, 1.0, 2.0],
    "CR": [0.2, 0.5, 1.0],
}
for F in parameters["F"]:
    for CR in parameters["CR"]:
        result = differential_evolution(sphere, bounds=BOUNDS, pop_size=50, F=F, CR=CR)
        animate_de(func=sphere, history=result.history, filename=f"F{F}_CR{CR}.gif")

Animation saved to F0.2_CR0.2.gif
Animation saved to F0.2_CR0.5.gif
Animation saved to F0.2_CR1.0.gif
Animation saved to F1.0_CR0.2.gif
Animation saved to F1.0_CR0.5.gif
Animation saved to F1.0_CR1.0.gif
Animation saved to F2.0_CR0.2.gif
Animation saved to F2.0_CR0.5.gif
Animation saved to F2.0_CR1.0.gif


In animations saved we can see that the Scale Factor is responsible for controlling the exploration of the search space, while the Crossover Rate is responsible for controlling the exploitation of the search space.<br>


### SHADE: Success-History Based Adaptive Differential Evolution

SHADE is an advanced variant of DE designed to enhance optimization performance through adaptive parameter control. Your task is to critically analyze the research paper [Success-History Based Parameter Adaptation for Differential Evolution](https://metahack.org/CEC2013-SHADE.pdf).

- Summarize the key concepts and mechanisms introduced in SHADE.
- Explain how SHADE differs from the standard DE algorithm.
- Discuss the motivation behind these changes and the empirical improvements demonstrated in the study.


##### Key Concepts
SHADE introduces a mechanism for adaptive parameter control based on the success history of previous generations. It maintains a memory of successful parameter settings and uses this information to adaptively adjust the parameters for future generations. This allows SHADE to dynamically adapt to the characteristics of the optimization problem, improving convergence speed and solution quality.

##### Differences from Standard DE
SHADE differs from standard DE in several ways:
1. **Adaptive Parameter Control**: SHADE uses a success-history based approach to adaptively adjust the parameters (F and CR) based on the performance of previous generations, while standard DE uses fixed parameters.
2. **Memory Mechanism**: SHADE maintains a memory of successful parameter settings, allowing it to learn from past experiences and improve future performance.
3. **Diverse Population**: SHADE employs a diverse population strategy to enhance exploration and exploitation, while standard DE relies on a fixed population size and structure.
##### Motivation Behind Changes

in each generation, JADE continuously updates µCR, µF such that they approach SCR, SF , which are the mean values for CR and F that have been successful in previous generations. While it is implicitly assumed that SCR and SF only includes parameter values which perform
well on the given problem, due to the probabilistic nature of DE, it is possible that poor settings for CR and F are also included in SCR and SF . If so, it can cause µCR, µF to move towards undesirable values, resulting in degraded search performance.

In order to improve upon the robustness of JADE, Success-History based Adaptive DE (SHADE) was proposed, an improved version of JADE which uses a different parameter adaptation mechanism based on a historical record of successful parameter settings. In SHADE, the mean values of SCR, SF for each generation are stored in a historical memory MCR, MF . In contrast to JADE, which uses a single pair (µCR, µF ) to guide parameter adaptation, SHADE maintains a diverse set of parameters to guide control parameter adaptation as search progresses. Thus, even if SCR, SF for
some particular generation contains a poor set of values, the parameters stored in memory from previous generations can not be directly, negatively impacted. This should result in SHADE being more robust than JADE.

SHADE was shown to outperform previous, state-of-the-art DE algorithms on a large set of benchmark problems, including the CEC2013 benchmark set, as well as CEC2005 benchmarks and the set of 13 classical benchmark problems.

### Recommended Reading

1. https://pablormier.github.io/2017/09/05/a-tutorial-on-differential-evolution-with-python/
2. Das, Swagatam, and Ponnuthurai Nagaratnam Suganthan. [Differential evolution: A survey of the state-of-the-art.](https://i2pc.es/coss/Docencia/SignalProcessingReviews/Das2011.pdf)
3. Tanabe, Ryoji, and Alex Fukunaga. [Success-history based parameter adaptation for differential evolution.](https://metahack.org/CEC2013-SHADE.pdf)
